In [ ]:
# %% [Cell 0 — Setup and template]
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Shared course palette
PALETTE = ['#7BB3B2', '#65A6BD', '#C997AF', '#B8B0D3', '#F4CF97', '#98B9A0', '#F6DECD']

# Build the template once (Principle 5)
nso_template = go.layout.Template()
nso_template.layout = go.Layout(
    font=dict(family='Arial', size=13, color='#333'),
    title_font=dict(size=16, color='#222'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    colorway=PALETTE,
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor='lightgray', gridwidth=0.5),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    margin=dict(l=60, r=30, t=60, b=40),
)
pio.templates['nso'] = nso_template
pio.templates.default = 'nso'

# Clean toolbar + sensible PNG export
PLOTLY_CONFIG = {
    'displaylogo': False,
    'modeBarButtonsToRemove': ['select2d', 'lasso2d', 'autoScale2d'],
    'toImageButtonOptions': {'format': 'png', 'width': 1200, 'height': 700, 'scale': 2},
}

### Cell 1 — Data preparation *(given, just run it)*

**EN** — We use only the **Q3 2025** file (`IEA_III_TRIMESTRE_2025.sav`), one row per individual. This cell selects a handful of variables, renames them to short English names, converts the numeric ones, and derives `employed`, `unemployed`, `age_group` and a short education label. Read it carefully: **Principle 4 says data preparation is the real job** — every chart below is built from this single tidy `wap` DataFrame.

**PT** — Usamos apenas o ficheiro do **III Trimestre 2025** (`IEA_III_TRIMESTRE_2025.sav`), com uma linha por indivíduo. Esta célula seleciona algumas variáveis, renomeia-as para nomes curtos em inglês, converte as numéricas e cria `employed`, `unemployed`, `age_group` e um rótulo curto de escolaridade. Leia com atenção: **o Princípio 4 diz que a preparação dos dados é o verdadeiro trabalho** — todos os gráficos abaixo saem deste único DataFrame `wap`.

In [ ]:
# %% [Cell 1 — Data preparation]
DATA_ROOT =
DATA_3_TRIMESTER = 'IEA_III_TRIMESTRE_2025.sav'
DATA_Q3 = os.path.join(DATA_ROOT, DATA_3_TRIMESTER)

q3_raw = pd.read_spss(DATA_Q3, convert_categoricals=True)
print('Raw shape:', q3_raw.shape)

Q3_COLS = {
    'PROV': 'province',
    'AREA_RESID': 'area',
    'S02_01': 'sex',
    'S02_02': 'age',
    'S03_03': 'edu_level_raw',
    'S5_03': 'employer_type',
    'S5_09': 'social_security',
    'S5_10': 'contract',
    'S5_24': 'income',
    'HORAS_HABITUAIS_TOTAL': 'hours_usual',
    'HORAS_EFECTIVAS_TOTAL': 'hours_actual',
    'S4_01': 'work_paid',
    'S4_02': 'work_own',
    'S4_03': 'work_family',
    'S4_09': 'absent_job',
    'S8_01': 'searched',
    'S8_12': 'available',
    'POND_IEA_III_TRIM_2025_IND': 'weight',
}

df = q3_raw[list(Q3_COLS)].rename(columns=Q3_COLS).copy()

TEXT_COLS = ['province', 'area', 'sex', 'edu_level_raw', 'employer_type',
             'social_security', 'contract', 'work_paid', 'work_own',
             'work_family', 'absent_job', 'searched', 'available']
NUM_COLS = ['age', 'income', 'hours_usual', 'hours_actual', 'weight']

for c in TEXT_COLS:
    df[c] = df[c].astype('string').str.strip()
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors='coerce')


def is_yes(s: pd.Series) -> pd.Series:
    """Angola IEA answers are labelled text: 'Sim' / 'Não'."""
    return s.str.upper().str.startswith('SIM').fillna(False)


df['employed'] = (
    is_yes(df['work_paid']) | is_yes(df['work_own'])
    | is_yes(df['work_family']) | is_yes(df['absent_job'])
)
df['unemployed'] = (~df['employed']) & is_yes(df['searched']) & is_yes(df['available'])
df['in_labour_force'] = df['employed'] | df['unemployed']

wap = df[df['age'].between(15, 99)].copy()

AGE_BINS = [15, 25, 35, 45, 55, 65, 100]
AGE_LABELS = ['15-24', '25-34', '35-44', '45-54', '55-64', '65+']
wap['age_group'] = pd.cut(wap['age'], bins=AGE_BINS, labels=AGE_LABELS, right=False)


def short_edu(label):
    if pd.isna(label):
        return 'Unknown'
    t = str(label).lower()
    if 'superior' in t:
        return 'Tertiary'
    if 'ii' in t and 'ciclo' in t:
        return 'Upper secondary'
    if 'ciclo' in t:
        return 'Lower secondary'
    if 'prim' in t or 'inicia' in t:
        return 'Primary'
    return 'None / other'


EDU_ORDER = ['None / other', 'Primary', 'Lower secondary', 'Upper secondary', 'Tertiary', 'Unknown']
wap['edu'] = pd.Categorical(wap['edu_level_raw'].map(short_edu), categories=EDU_ORDER, ordered=True)

SEX_ORDER = ['Masculino', 'Feminino']
SEX_EN = {'Masculino': 'Male', 'Feminino': 'Female'}


def wshare(frame, flag_col, weight_col='weight'):
    """Weighted share (%) of rows where flag_col is True."""
    f = frame.dropna(subset=[weight_col])
    if len(f) == 0 or f[weight_col].sum() == 0:
        return np.nan
    return 100 * np.average(f[flag_col].astype(float), weights=f[weight_col])


print('Working-age rows:', len(wap))
print('Employment rate (weighted, 15+): {:.1f}%'.format(wshare(wap, 'employed')))
wap[['province', 'area', 'sex', 'age', 'age_group', 'edu', 'employed', 'income', 'weight']].head()

### Task 1 — Interviews by province (simple bar)

**EN — build this**

- A **vertical bar chart**: one bar per province, height = number of working-age individuals interviewed there, sorted from most to fewest.
- Title, y-axis title, readable x labels (18 provinces do not fit horizontally).
- Structure to respect: **one `go.Bar` trace inside a `go.Figure`, everything else in `fig.update_layout()`** (**Principle 1**).

*Hints*

- Counting the occurrences of each value in a column is a single pandas method — and it returns the result **already sorted**. Type `wap['province'].` + `Tab` and look for it.
- A Series has `.index` (the labels) and `.values` (the numbers). Traces want actual arrays, not column names.
- Long category labels: look for a layout key starting with `xaxis_tick...`.
- Optional polish: `text=` and `textposition='outside'` on the trace print the value on top of each bar.

**PT — construa isto**

- Um **gráfico de barras verticais**: uma barra por província, altura = número de indivíduos em idade activa entrevistados, ordenado do maior para o menor.
- Título, título do eixo y e rótulos legíveis no eixo x (18 províncias não cabem na horizontal).
- Estrutura a respeitar: **um trace `go.Bar` dentro de um `go.Figure`; todo o resto em `fig.update_layout()`** (**Princípio 1**).

*Dicas*

- Contar as ocorrências de cada valor numa coluna é um único método do pandas — e devolve o resultado **já ordenado**. Escreva `wap['province'].` + `Tab` e procure-o.
- Uma Series tem `.index` (rótulos) e `.values` (números). Os traces querem arrays de dados, não nomes de colunas.
- Rótulos compridos: procure uma chave de layout que comece por `xaxis_tick...`.
- Extra opcional: `text=` e `textposition='outside'` no trace escrevem o valor no topo de cada barra.

In [ ]:
# %% [Task 1 — Interviews by province]
# ----- Step 1: Prepare -----
# Build a Series: index = province, values = number of interviewed individuals,
# sorted descending. Print it before plotting.


# ----- Step 2: Plot -----
# go.Figure(go.Bar(...)) -> fig.update_layout(...) -> fig.show(config=PLOTLY_CONFIG)

### Task 2 — Age distribution and where people live (histogram + donut)

**EN — build this**

- **2a.** A **histogram** of `age` for the working-age population, around 25 bins, plus a dashed vertical line at the **median age** with a label showing the value. The line is layout chrome, not a trace (**Principle 3**).
- **2b.** A **donut chart** of the urban/rural split. Report the **estimated population**, i.e. use the survey `weight`, not row counts.
- **2c.** Print a small two-column table comparing the *unweighted sample* share with the *weighted population* share of urban/rural, and write one sentence in a markdown cell explaining the difference.

*Hints*

- Median of a Series: `.median()`. Reference line: `fig.add_vline(x=..., line_dash='dash', line_color='red', annotation_text=...)` — build the label with an f-string such as `f'Median: {median_age:.0f} years'`.
- Bin count is a trace property of `go.Histogram`, spelled `nbinsx`.
- Estimated population per area = **sum of the weights** within each area: `groupby(...)[...]. sum()`. Pass `observed=True` when grouping on categoricals to keep pandas quiet.
- `go.Pie` wants `labels=` and `values=`; `hole=0.4` turns a pie into a donut, and `textinfo='percent+label'` removes the need for a legend.
- For the comparison table: `value_counts(normalize=True)` gives sample shares; `pd.concat([...], axis=1)` puts the two Series side by side.

**PT — construa isto**

- **2a.** Um **histograma** da variável `age` para a população em idade activa, com cerca de 25 classes, mais uma linha vertical tracejada na **idade mediana**, com rótulo a mostrar o valor. A linha é decoração do *layout*, não um trace (**Princípio 3**).
- **2b.** Um **gráfico de rosca** da distribuição urbano/rural. Reporte a **população estimada**, ou seja, use o `weight` do inquérito e não a contagem de linhas.
- **2c.** Imprima uma pequena tabela de duas colunas comparando a proporção na *amostra sem ponderação* com a proporção da *população ponderada*, e escreva uma frase numa célula markdown a explicar a diferença.

*Dicas*

- Mediana de uma Series: `.median()`. Linha de referência: `fig.add_vline(x=..., line_dash='dash', line_color='red', annotation_text=...)` — construa o rótulo com uma f-string como `f'Median: {median_age:.0f} years'`.
- O número de classes é uma propriedade do trace `go.Histogram`, chamada `nbinsx`.
- População estimada por área = **soma dos ponderadores** dentro de cada área: `groupby(...)[...].sum()`. Use `observed=True` ao agrupar por categóricas.
- `go.Pie` recebe `labels=` e `values=`; `hole=0.4` transforma o círculo numa rosca e `textinfo='percent+label'` dispensa a legenda.
- Para a tabela: `value_counts(normalize=True)` dá as proporções da amostra; `pd.concat([...], axis=1)` junta as duas Series lado a lado.

In [ ]:
# %% [Task 2a — Age histogram with median reference line]
# ----- Step 1: Prepare -----
# Series of ages without missing values + the median value.


# ----- Step 2: Plot -----
# One go.Histogram trace, then add_vline(...), then update_layout(...).


# %% [Task 2b + 2c — Urban / rural population share (donut)]
# ----- Step 1: Prepare -----
# (i)  estimated population per area = sum of weights
# (ii) sample shares (%) and weighted shares (%) in one printed table


# ----- Step 2: Plot -----
# One go.Pie trace with hole=0.4.

### Task 3 — Population pyramid by sex and age group

**EN — build this**

- The classic **population pyramid**: `age_group` on the y-axis, males extending left, females extending right, values as **% of the total 15+ population** (weighted).
- Two horizontal `go.Bar` traces on one figure (**Principle 2**), age bands in ascending order from the bottom.
- The x-axis must read as positive percentages on both sides — no minus signs visible to the reader.

*Hints*

- Prepare a small table with rows = age group, columns = sex: group by **two** keys, sum the weights, then reshape long-to-wide with `.unstack('sex')`. `reindex(AGE_LABELS)` guarantees the order, `.fillna(0)` closes any gap.
- Percentages of the grand total: divide the whole table by `table.values.sum()` and multiply by 100.
- The pyramid trick is a **minus sign** on one of the two `x=` arrays. Nothing else changes.
- Bars must start from the same baseline, so this is `barmode='overlay'` (try `'group'` once to see why it is wrong). `bargap` controls the gap between age bands.
- Hiding the minus signs is a tick relabelling job: `xaxis=dict(tickvals=[-8, -6, ..., 8], ticktext=[...])` built with `abs()`.
- Bonus: `hovertemplate` plus `customdata` lets you show the positive value on hover for the negative side.

**PT — construa isto**

- A clássica **pirâmide populacional**: `age_group` no eixo y, homens para a esquerda, mulheres para a direita, valores em **% da população total de 15+ anos** (ponderada).
- Dois traces `go.Bar` horizontais na mesma figura (**Princípio 2**), com os grupos de idade em ordem crescente de baixo para cima.
- O eixo x deve mostrar percentagens positivas nos dois lados — sem sinais negativos visíveis para o leitor.

*Dicas*

- Prepare uma tabela com linhas = grupo de idade e colunas = sexo: agrupe por **duas** chaves, some os ponderadores e passe de formato longo a largo com `.unstack('sex')`. `reindex(AGE_LABELS)` garante a ordem e `.fillna(0)` fecha lacunas.
- Percentagens do total: divida a tabela inteira por `table.values.sum()` e multiplique por 100.
- O truque da pirâmide é um **sinal menos** num dos dois arrays `x=`. Mais nada muda.
- As barras têm de partir da mesma linha de base, logo `barmode='overlay'` (experimente `'group'` uma vez para ver porque está errado). `bargap` controla o espaço entre grupos.
- Esconder os sinais negativos é uma questão de rótulos: `xaxis=dict(tickvals=[-8, -6, ..., 8], ticktext=[...])`, construído com `abs()`.
- Bónus: `hovertemplate` com `customdata` permite mostrar o valor positivo no lado negativo.

In [ ]:
# %% [Task 3 — Population pyramid]
# ----- Step 1: Prepare -----
# Table: rows = AGE_LABELS, columns = sex, values = % of total weighted population.
# Print it and check that the whole table sums to ~100.


# ----- Step 2: Plot -----
# Two horizontal go.Bar traces (one negative), then the layout work:
# barmode, bargap, x tick relabelling, axis titles.

### Task 4 — Income and hours in one figure (box plots + scatter)

**EN — build this**

- A **1×2 subplot figure**:
    - left panel: **box plots** of net monthly `income` (main job) by education level, one box per level of `EDU_ORDER`, log-scaled y-axis;
    - right panel: a **scatter** of `hours_usual` (x) against `income` (y) for employed people, **one trace per sex** so the legend separates them, log-scaled y-axis.
- Keep only sensible records: positive income, positive hours.
- Then answer in a markdown cell: what does the log scale hide, and what do the extreme values tell you about how the income question was recorded?

*Hints*

- Skeleton: `fig = make_subplots(rows=1, cols=2, subplot_titles=(..., ...), column_widths=[0.45, 0.55])`, then every `add_trace(..., row=1, col=1)` / `(..., row=1, col=2)`.
- A box plot needs only the raw values: `go.Box(y=<array>, name=<label>)`. Loop over `EDU_ORDER` and skip levels with no observations (`if subset.empty: continue`).
- Scatter of dots is `go.Scatter(..., mode='markers')` — remember there is no `go.Line` and no `go.Scatterplot`. Overlapping points need `marker=dict(size=6, opacity=0.4)`.
- Log axes on a subplot: `fig.update_yaxes(type='log', row=..., col=...)`, one call per panel. Titles per axis: `title_text=`.
- `showlegend=False` on the boxes keeps the legend to just Male/Female.
- Useful sanity print before plotting: `inc.groupby('edu', observed=True)['income'].describe()`.

**PT — construa isto**

- Uma figura com **subplots 1×2**:
    - painel esquerdo: **diagramas de caixa** do `income` mensal líquido (emprego principal) por nível de escolaridade, uma caixa por nível de `EDU_ORDER`, com eixo y logarítmico;
    - painel direito: um **gráfico de dispersão** de `hours_usual` (x) contra `income` (y) para as pessoas empregadas, **um trace por sexo**, com eixo y logarítmico.
- Mantenha apenas registos razoáveis: rendimento positivo e horas positivas.
- Depois responda numa célula markdown: o que esconde a escala logarítmica e o que dizem os valores extremos sobre a forma como a pergunta do rendimento foi registada?

*Dicas*

- Estrutura: `fig = make_subplots(rows=1, cols=2, subplot_titles=(..., ...), column_widths=[0.45, 0.55])` e depois cada `add_trace(..., row=1, col=1)` / `(..., row=1, col=2)`.
- Uma caixa precisa apenas dos valores brutos: `go.Box(y=<array>, name=<rótulo>)`. Percorra `EDU_ORDER` e salte níveis sem observações (`if subset.empty: continue`).
- Dispersão de pontos é `go.Scatter(..., mode='markers')` — lembre-se: não existe `go.Line`. Pontos sobrepostos pedem `marker=dict(size=6, opacity=0.4)`.
- Eixos logarítmicos em subplots: `fig.update_yaxes(type='log', row=..., col=...)`, uma chamada por painel. Títulos dos eixos: `title_text=`.
- `showlegend=False` nas caixas mantém a legenda apenas com Masculino/Feminino.
- Verificação útil antes de desenhar: `inc.groupby('edu', observed=True)['income'].describe()`.

In [ ]:
# %% [Task 4 — Income by education + hours vs income]
# ----- Step 1: Prepare -----
# (i)  income sample: employed-or-not, income > 0
# (ii) scatter sample: employed, income > 0, hours_usual > 0
# Print a describe() table by education level before plotting.


# ----- Step 2: Plot -----
# make_subplots(1x2) -> loop of go.Box on the left -> loop of go.Scatter on the right
# -> update_layout / update_xaxes / update_yaxes (log!) -> fig.show(config=PLOTLY_CONFIG)

### Task 5 — Employment rate: grouped bars + heatmap

**EN — build this**

- **5a.** The **weighted employment rate** of the 15+ population by **province × sex**, as a **grouped bar chart** (one trace per sex), with provinces ordered by their overall rate — highest on the left.
- **5b.** The same indicator by **province × age group**, as a **heatmap** with the value printed in each cell.
- Interpretation in a markdown cell: which provinces and which age groups pull the national average up or down? Is the youth pattern visible?

*Hints*

- A rate is not a count, so `value_counts()` will not help. You need `wshare()` from Cell 1 applied **within each group**: `wap.groupby(keys, observed=True).apply(lambda g: wshare(g, 'employed'))`. Write it once as a tiny helper `rate_by(frame, keys)` and reuse it three times.
- With one key you get a Series indexed by province — that is your ordering: `.sort_values(ascending=False)`. With two keys, `.unstack(<second key>)` turns it into the province × sex table, and `.reindex(prov_order.index)` applies the ordering.
- If pandas warns about grouping columns inside `apply`, add `include_groups=False` (pandas ≥ 2.2) or ignore it.
- Grouped bars: one `go.Bar` per sex, `x=` provinces, `y=` that sex's column, and `barmode='group'` in the layout. Try `'stack'` once and write down in one line why stacking rates is meaningless.
- `go.Heatmap` needs a **2-D array** in `z=` (your table's `.values`) plus `x=` column labels and `y=` row labels as lists. Printed cell values: `text=` with `texttemplate='%{text:.0f}'`. Reindex rows/columns so the axes are ordered on purpose, and give the colour bar a title with `colorbar=dict(title=...)`.
- 18 provinces need vertical room: `height=` in the layout.

**PT — construa isto**

- **5a.** A **taxa de emprego ponderada** da população de 15+ anos por **província × sexo**, num **gráfico de barras agrupadas** (um trace por sexo), com as províncias ordenadas pela taxa global — a mais alta à esquerda.
- **5b.** O mesmo indicador por **província × grupo de idade**, num **mapa de calor** com o valor escrito em cada célula.
- Interpretação numa célula markdown: que províncias e que grupos de idade puxam a média nacional para cima ou para baixo? O padrão dos jovens é visível?

*Dicas*

- Uma taxa não é uma contagem, por isso `value_counts()` não serve. Precisa de aplicar `wshare()` da Célula 1 **dentro de cada grupo**: `wap.groupby(keys, observed=True).apply(lambda g: wshare(g, 'employed'))`. Escreva uma pequena função `rate_by(frame, keys)` e reutilize-a três vezes.
- Com uma chave obtém uma Series indexada por província — essa é a sua ordenação: `.sort_values(ascending=False)`. Com duas chaves, `.unstack(<segunda chave>)` cria a tabela província × sexo e `.reindex(prov_order.index)` aplica a ordenação.
- Se o pandas avisar sobre as colunas de agrupamento dentro do `apply`, acrescente `include_groups=False` (pandas ≥ 2.2) ou ignore.
- Barras agrupadas: um `go.Bar` por sexo, `x=` províncias, `y=` a coluna desse sexo e `barmode='group'` no layout. Experimente `'stack'` uma vez e escreva numa linha por que empilhar taxas não faz sentido.
- `go.Heatmap` precisa de um **array 2-D** em `z=` (os `.values` da sua tabela), mais `x=` com os rótulos das colunas e `y=` com os das linhas, em listas. Valores nas células: `text=` com `texttemplate='%{text:.0f}'`. Reindexe linhas e colunas para ordenar os eixos e dê título à barra de cores com `colorbar=dict(title=...)`.
- 18 províncias exigem espaço vertical: `height=` no layout.

In [ ]:
# %% [Task 5a — Employment rate by province and sex]
# ----- Step 1: Prepare -----
# helper rate_by(frame, keys) -> weighted employment rate per group
# (i)  prov_order : Series, province -> overall rate, sorted descending
# (ii) rate_sex   : DataFrame, rows = provinces (ordered), columns = sex


# ----- Step 2: Plot -----
# One go.Bar per sex + barmode='group'.


# %% [Task 5b — Employment rate heatmap: province x age group]
# ----- Step 1: Prepare -----
# heat : DataFrame, rows = provinces (same order as 5a), columns = AGE_LABELS


# ----- Step 2: Plot -----
# One go.Heatmap trace with z / x / y / text / texttemplate / colorscale.

### Task 6 — Advanced: compare Q3 and Q4 (harmonise two files)

**EN — build this**

- The Q4 file (`data/IEA_2025_IV_TRIM_IND.sav`) measures the same concepts under **different variable names**. Harmonise it against your Q3 frame, stack the two quarters, then compare.
- Steps:
    1. Load Q4 and build your own rename map for: province, area, sex, age, the three "did you work?" questions, "absent from a job", and the individual weight. The naming conventions table on Employment Survey tells you which prefixes to look in (`DEM_*`, `ATW_*`, `ABS_*`, `POND_*`).
    2. Rebuild the same `employed` flag with the same logic and the same `is_yes()` helper, restrict to 15+, add a `quarter` column to both frames, keep a common list of columns, and stack them.
    3. **Verify the harmonisation before plotting**: for `province`, `sex` and `area`, compare the label sets across quarters and print the labels that appear in only one of them. Fix real mismatches with a mapping dictionary; document any you decide to leave alone.
    4. **Chart A** — grouped bar: employment rate by `sex` × `quarter`.
    5. **Chart B** — **dumbbell chart** by province: one thin grey line per province from its Q3 value to its Q4 value, plus one marker trace for Q3 and one for Q4. Sort provinces by the size of the change so the story reads top-to-bottom.
    6. In a markdown cell, list what is **not** comparable between the two quarters and why (start with income and with the job-search reference period).

*Hints*

- Set operations answer "which labels differ?": `set(a) ^ set(b)` is the symmetric difference. `.replace(mapping)` applies your fixes.
- Stacking two frames with the same columns: `pd.concat([...], ignore_index=True)`. Tag each with `assign(quarter=...)` first.
- Reuse `rate_by()` from Task 5 — only the keys change (`['sex', 'quarter']`, `['province', 'quarter']`). `.unstack('quarter')` gives you one column per quarter; the change is then a simple column subtraction, and `sort_values` orders the chart.
- The dumbbell has no dedicated trace type. It is a **loop** of `go.Scatter(x=[q3_value, q4_value], y=[province, province], mode='lines')` with `showlegend=False, hoverinfo='skip'`, plus two `mode='markers'` traces on top. Iterate rows with `.iterrows()`.
- Careful with `dropna()`: a province present in only one quarter would otherwise draw a half-line.

**PT — construa isto**

- O ficheiro do IV Trimestre (`data/IEA_2025_IV_TRIM_IND.sav`) mede os mesmos conceitos com **nomes de variáveis diferentes**. Harmonize-o com o seu quadro do Q3, junte os dois trimestres e compare.
- Passos:
    1. Carregue o Q4 e construa o seu próprio dicionário de renomeação para: província, área, sexo, idade, as três perguntas "trabalhou?", "ausente do emprego" e o ponderador individual. A tabela de convenções de nomes em Employment Survey indica os prefixos a consultar (`DEM_*`, `ATW_*`, `ABS_*`, `POND_*`).
    2. Reconstrua a mesma variável `employed` com a mesma lógica e a mesma função `is_yes()`, restrinja a 15+, acrescente uma coluna `quarter` a ambos os quadros, mantenha uma lista comum de colunas e junte-os.
    3. **Verifique a harmonização antes de desenhar**: para `province`, `sex` e `area`, compare os conjuntos de rótulos entre trimestres e imprima os que aparecem só num deles. Corrija as diferenças reais com um dicionário de correspondência e documente as que decidir manter.
    4. **Gráfico A** — barras agrupadas: taxa de emprego por `sex` × `quarter`.
    5. **Gráfico B** — **gráfico de haltere** por província: uma linha cinzenta fina por província, do valor do Q3 ao do Q4, mais um trace de marcadores para o Q3 e outro para o Q4. Ordene as províncias pela dimensão da variação.
    6. Numa célula markdown, liste o que **não** é comparável entre os dois trimestres e porquê (comece pelo rendimento e pelo período de referência da procura de emprego).

*Dicas*

- Operações com conjuntos respondem à pergunta "que rótulos diferem?": `set(a) ^ set(b)` é a diferença simétrica. `.replace(mapping)` aplica as correções.
- Juntar dois quadros com as mesmas colunas: `pd.concat([...], ignore_index=True)`. Marque cada um com `assign(quarter=...)` antes.
- Reutilize `rate_by()` da Tarefa 5 — só mudam as chaves (`['sex', 'quarter']`, `['province', 'quarter']`). `.unstack('quarter')` dá uma coluna por trimestre; a variação é uma subtração de colunas e `sort_values` ordena o gráfico.
- O haltere não tem tipo de trace próprio. É um **ciclo** de `go.Scatter(x=[valor_q3, valor_q4], y=[província, província], mode='lines')` com `showlegend=False, hoverinfo='skip'`, mais dois traces `mode='markers'` por cima. Percorra as linhas com `.iterrows()`.
- Atenção ao `dropna()`: uma província presente só num trimestre desenharia meia linha.

In [ ]:
# %% [Task 6 — Harmonise Q3 + Q4 and compare]
# ----- Step 1: Prepare -----
# (i)   load Q4, build your rename map, cast types, rebuild `employed`, filter 15+
# (ii)  tag both quarters and stack them into one `panel` frame
# (iii) print the label sets that differ across quarters, then fix them
# (iv)  rate by sex x quarter, and rate by province x quarter with a `change` column


# ----- Step 2: Plot — Chart A (grouped bar by sex) -----


# ----- Step 2: Plot — Chart B (dumbbell by province) -----